## Divide into Chunks — We Follow the Below Approach

### 1. Chunking methodology

We follow a **hybrid Markdown header-based and recursive text-splitting approach** to process our 14 Markdown knowledge-base documents.

- **Stage 1 — Header-based splitting:** Divide documents into meaningful sections using Markdown headings (`#`, `##`, `###`).
- **Stage 2 — Recursive splitting:** Further divide large sections while preserving natural text boundaries.
- **Character-based splitting:** Measure chunk size and overlap in characters.
- **Token-aware splitting:** Measure chunk size and overlap using the tokenizer associated with the selected embedding model.

Each chunk retains its source filename, category, heading metadata and chunk index.

### 2. Creating 10 vector collections

We use **ChromaDB**, a vector database that stores embeddings and supports similarity search.

**Vector database vs. collection:**
- **Vector database:** The overall storage system that manages embeddings, indexes and metadata.
- **Collection:** A separate logical grouping of embeddings within the database. Each collection can use a different embedding model or chunking configuration.

We use **one ChromaDB database containing 10 independent collections**, created from:

**5 embedding models × 2 chunking methods = 10 collections**

| Embedding model | Character-based collection | Token-aware collection |
|---|---|---|
| all-MiniLM-L6-v2 | `minilm_char` | `minilm_token` |
| BGE Small v1.5 | `bge_small_char` | `bge_small_token` |
| BGE Base v1.5 | `bge_base_char` | `bge_base_token` |
| Mixedbread Large v1 | `mxbai_large_char` | `mxbai_large_token` |
| OpenAI 3 Large | `openai_large_char` | `openai_large_token` |

**Initial baseline configurations:**

| Splitting method | Chunk size | Overlap |
|---|---:|---:|
| Character-based | 700 characters | 80 characters |
| Token-aware | 180 tokens | 20 tokens |

The token-aware splitter uses each embedding model's corresponding tokenizer. All embedding inputs must respect the selected model's token limit.

We evaluate the 10 configurations against our 500-question evaluation dataset, comparing retrieval quality, accuracy across question categories, latency and resource requirements.

### 3. Further chunk-size and overlap optimisation

After baseline evaluation, we shortlist the **best 2–3 embedding/chunking candidates**, considering retrieval accuracy and computational cost.

For these selected embedding models, we experiment with three chunk-size/overlap configurations per splitting method:

| Experiment | Character-based | Token-aware |
|---|---|---|
| Small | 500 / 50 | 120 / 12 |
| Medium (baseline) | 700 / 80 | 180 / 20 |
| Large | 900 / 100 | 210 / 20 |

These are starting configurations. Larger token sizes may additionally be evaluated for models supporting longer inputs. The actual token count, including heading context, must be checked to prevent truncation.

Each unique combination of embedding model, splitting method, chunk size and overlap is stored in an independently named Chroma collection, allowing reproducible comparisons.

### 4. Total planned collections

Assuming we shortlist **3 embedding models** and test three parameter configurations per splitting method:

| Phase | Collections |
|---|---:|
| Stage 1 — Initial baseline (5 models × 2 methods) | 10 |
| Stage 2 — Additional tuning (3 shortlisted models × 2 methods × 2 new configurations) | 12 |
| **Total unique collections** | **22** |

If we shortlist only 2 models, the total would be **18 collections**.

A complete experiment applying all three configurations to all five models would require **30 collections**.

If we independently test every combination of three chunk sizes and two overlap settings for all five models and both splitting methods, the full grid would require **60 collections**.

### 5. Recommended approach

We recommend the **staged evaluation strategy with approximately 18–22 collections**, rather than exhaustively testing every possible combination.

The process is:

**Build 10 baseline collections → Evaluate retrieval performance → Shortlist 2–3 models → Optimise chunk sizes and overlaps → Select the best configurations → Apply advanced RAG techniques.**

Techniques such as reranking, hybrid search, query rewriting and Top-K optimisation can subsequently be tested on shortlisted configurations without necessarily rebuilding their vector collections.

This approach reduces unnecessary computation and API costs while providing a systematic, measurable and reproducible RAG optimisation process.

In [1]:
import glob 
import tiktoken
import numpy as np
from dotenv import load_dotenv
from pathlib import Path

In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

/tmp/ipykernel_29142/2075362422.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [3]:
import os
import gc
import chromadb

from getpass import getpass
from transformers import AutoTokenizer

from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter
)
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
# How many files in all the documents?

knowledge_base_path = "../backend/KnowledgeBase/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")
 

Found 14 files in the knowledge base


In [5]:
# How many characters in all the documents?

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Total characters in knowledge base: 53,878


In [6]:
# Load in everything in the knowledgebase using LangChain's loaders

#folders = glob.glob("knowledge_base/*")
documents = []
for file in files:
    doc_type = os.path.basename(file)
    #print(f"folder :{file} , doc_cat : {doc_type}")
    #loader = DirectoryLoader(file, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    loader = TextLoader(file)
    loader_docs = loader.load()
    for doc in loader_docs:
        #doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 14 documents


In [7]:
# Separate storage for our evaluation database
EVAL_DB_PATH = "./chroma_eval_db"

# Folder for downloaded local embedding models
MODEL_CACHE_DIR = "./embedding_models"

# OpenAI API key (required for OpenAI embedding model)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter OpenAI API key: "
    )


### Create our reusable splitter function - if model is provided it means it is a token based splitting.

if no model provided means text based splitting. 

In [8]:

def get_text_splitter(chunk_size, chunk_overlap, model_name=None):

    # No model name means character-based splitting
    if model_name is None:

        return RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

    # OpenAI model: use tiktoken
    if model_name.startswith("text-embedding-3-"):

        return RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            model_name=model_name,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

    # Other models: use Hugging Face tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    return RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )


### Create chunks from the knowledge base - This function accepts your existing LangChain documents list.

In [9]:

def create_chunks(documents, chunk_size, chunk_overlap, model_name=None):

    # Stage 1: Split Markdown using headings
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "heading_1"),
            ("##", "heading_2"),
            ("###", "heading_3")
        ],
        strip_headers=False
    )

    # Stage 2: Character or token-aware splitting
    text_splitter = get_text_splitter(
        chunk_size,
        chunk_overlap,
        model_name
    )

    all_chunks = []

    # Process one Markdown document at a time
    for doc in documents:

        sections = header_splitter.split_text(doc.page_content)

        # If no sections are found, use original document
        if not sections:
            sections = [doc]

        chunk_index = 0

        for section in sections:

            # Preserve original filename and document category
            # while keeping Markdown heading metadata
            section.metadata.update(doc.metadata)

            # Split long sections into smaller pieces
            smaller_chunks = text_splitter.split_documents([section]) # this does the actual splitting based on cahracted or token

            for chunk in smaller_chunks:

                # Number the chunks within each original document
                chunk.metadata["chunk_index"] = chunk_index

                all_chunks.append(chunk)
                chunk_index += 1

    return all_chunks


### Test the chunking function first -Before creating any embeddings, I recommend testing the function with just one configuration.

In [10]:
char_chunks = create_chunks(
    documents,
    chunk_size=700,
    chunk_overlap=80
)

print("Original documents:", len(documents))
print("Generated chunks:", len(char_chunks))

# Examine one sample chunk
print("\nSample chunk:\n")
print(char_chunks[0].page_content)

print("\nMetadata:")
print(char_chunks[0].metadata)

Original documents: 14
Generated chunks: 128

Sample chunk:

# Current Learning, Completed Hands-On Work and Professional Development — Tonu James  
## Current professional direction
Tonu is transitioning from established enterprise software engineering to applied AI and AI engineering, following his completed MSc in Human-Centred Artificial Intelligence at TU Dublin. He is interested in three related directions without ranking one above the others: Generative AI/LLM applications, Computer Vision/Edge AI and Enterprise AI/SAP Generative AI.

Metadata:
{'heading_1': 'Current Learning, Completed Hands-On Work and Professional Development — Tonu James', 'heading_2': 'Current professional direction', 'source': '../backend/KnowledgeBase/Technical expertise and learning/current_learning.md', 'chunk_index': 0}


In [11]:
# "minilm": "sentence-transformers/all-MiniLM-L6-v2",
#    "mxbai_large": "mixedbread-ai/mxbai-embed-large-v1",
#    "openai_large": "text-embedding-3-large"
# bge_small": "BAAI/bge-small-en-v1.5",
# "bge_base": "BAAI/bge-base-en-v1.5",

models = {  "minilm": "sentence-transformers/all-MiniLM-L6-v2",
            "mxbai_large": "mixedbread-ai/mxbai-embed-large-v1"
    }

# Initial baseline splitting parameters
CHAR_SIZE = 700
CHAR_OVERLAP = 80

TOKEN_SIZE = 180
TOKEN_OVERLAP = 20

# Create character chunks once because they do not
# depend on the embedding model or its tokenizer
char_chunks = create_chunks(
    documents,
    chunk_size=CHAR_SIZE,
    chunk_overlap=CHAR_OVERLAP
)

print("Character-based chunks:", len(char_chunks))


Character-based chunks: 128


In [12]:

# Open the local ChromaDB database
chroma_client = chromadb.PersistentClient(path=EVAL_DB_PATH)


# Process one embedding model at a time
for model_key, model_name in models.items():

    print(f"\nProcessing model: {model_name}")

    # Step 1: Load embedding model
    if model_name == "text-embedding-3-large":

        # OpenAI API-based embeddings
        embedding_model = OpenAIEmbeddings(
            model=model_name
        )

    else:

        # Free local embedding model using FastEmbed
        embedding_model = FastEmbedEmbeddings(
            model_name=model_name,
            cache_dir=MODEL_CACHE_DIR,
            batch_size=32
        )

    # Step 2: Process both splitting methods
    for method in ["char", "token"]:

        if method == "char":

            chunks = char_chunks
            chunk_size = CHAR_SIZE
            chunk_overlap = CHAR_OVERLAP

        else:

            # Token-aware chunks depend on model tokenizer
            chunks = create_chunks(
                documents,
                chunk_size=TOKEN_SIZE,
                chunk_overlap=TOKEN_OVERLAP,
                model_name=model_name
            )

            chunk_size = TOKEN_SIZE
            chunk_overlap = TOKEN_OVERLAP

        # Step 3: Create a unique collection name
        collection_name = f"{model_key}_{method}"

        print(f"\nCreating collection: {collection_name}")
        print(f"Number of chunks: {len(chunks)}")

        # Step 4: Remove only this evaluation collection
        # if it already exists, so we can rebuild it cleanly
        existing_names = [
            c.name for c in chroma_client.list_collections()
        ]

        if collection_name in existing_names:
            chroma_client.delete_collection(
                name=collection_name
            )

        # Step 5: Create a Chroma collection
        vector_db = Chroma(
            collection_name=collection_name,
            embedding_function=embedding_model,
            persist_directory=EVAL_DB_PATH,

            # Use cosine distance for all collections
            collection_configuration={
                "hnsw": {"space": "cosine"}
            },

            # Save settings for future reference
            collection_metadata={
                "embedding_model": model_name,
                "split_method": method,
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap
            }
        )

        # Step 6: Generate embeddings and save in Chroma
        vector_db.add_documents(chunks)

        print(f"Saved collection: {collection_name}")

        del vector_db

    # Release this embedding model before loading the next
    del embedding_model
    gc.collect()


print("\nAll 10 collections created successfully!")



Processing model: sentence-transformers/all-MiniLM-L6-v2


2026-11-01 23:19:57.131880342 [W:onnxruntime:Default, device_discovery.cc:95 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+



Creating collection: minilm_char
Number of chunks: 128
Saved collection: minilm_char

Creating collection: minilm_token
Number of chunks: 122
Saved collection: minilm_token

Processing model: mixedbread-ai/mxbai-embed-large-v1


Fetching 5 files: 100%|██████████| 5/5 [00:21<00:00,  4.37s/it]



Creating collection: mxbai_large_char
Number of chunks: 128


: 